# Обученные лучи и интерактивные графики

Пакет устанавливается из GitHub. EMA-веса после 120 эпох и контрольная выборка
загружаются из отдельных файлов `demo/` — из локального checkout или через clone.
В ноутбуке нет встроенных wheel и весов. Выполните ячейки сверху вниз.


In [ ]:
# Установить пакет из GitHub в Python текущего Jupyter kernel.
import subprocess
import sys

REPOSITORY_URL = "https://github.com/Vovanm88/ddpm_vibe_research.git"
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
    "diffusion-lab[interactive] @ git+" + REPOSITORY_URL,
])


In [ ]:
# Веса — отдельные файлы demo/pipeline в репозитории.
import subprocess
import tempfile
from pathlib import Path

PROJECT_ROOT = next(
    (root for root in (Path.cwd(), *Path.cwd().parents)
     if (root / "demo/pipeline/model_index.json").is_file()),
    None,
)
if PROJECT_ROOT is None:
    PROJECT_ROOT = Path(tempfile.mkdtemp(prefix="diffusion-lab-demo-")) / "repository"
    subprocess.check_call([
        "git", "clone", "--depth", "1", REPOSITORY_URL, str(PROJECT_ROOT),
    ])
PIPELINE_PATH = PROJECT_ROOT / "demo/pipeline"
REFERENCE_PATH = PROJECT_ROOT / "demo/reference.npz"
print("Model files:", PIPELINE_PATH)


In [ ]:
import numpy as np
import torch
from IPython.display import display
try:
    from google.colab import output as colab_output
    colab_output.enable_custom_widget_manager()
except ImportError:
    pass
from diffusion_lab import ResearchDiffusionPipeline
from diffusion_lab.environments import SampleDataset
from diffusion_lab.visualization import notebook_field_explorer, notebook_sampling_explorer

torch.set_num_threads(4)
pipeline = ResearchDiffusionPipeline.from_pretrained(
    PIPELINE_PATH, local_files_only=True, low_cpu_mem_usage=False,
).to("cpu")
with np.load(REFERENCE_PATH, allow_pickle=False) as reference:
    dataset = SampleDataset(reference["samples"], reference["conditions"], num_classes=4,
                            metadata={"name": "rotating_rays", "seed": 124})
print(type(pipeline.model).__name__, type(pipeline.scheduler).__name__)


## Поле: condition, timestep, CFG, potential

Слайдеры пересчитывают поле по модели при отпускании ползунка. Цвет стрелок
кодирует исходную величину; нормировка меняет только отображение.
`Recover potential` решает F≈−∇U по исходным векторам и показывает residual/curl.
Ненулевой curl означает, что восстановленная U является проекцией.


In [ ]:
fields = notebook_field_explorer(pipeline, dataset, resolution=32,
                                 timestep=349, guidance_scale=1.5)
fields.controls["condition"].value = 1  # Class 0; 0 означает Unconditional
fields.controls["potential"].value = True
display(fields)


## Генерация и реальные кадры денойзинга

Выберите condition / CFG / steps / seed и нажмите **Generate**. Play и Frame
проигрывают сохранённые состояния этого же запуска; перемещение Frame не
генерирует новый шум. Контролы не запускают обучение в фоне.


In [ ]:
sampling = notebook_sampling_explorer(pipeline, dataset, num_samples=256,
                                       steps=100, seed=127, guidance_scale=1.5)
sampling.generate()
display(sampling)


## Использовать контролы в своём исследовании

После обычного PyTorch обучения передайте свой `ResearchDiffusionPipeline`:
```python
fields = notebook_field_explorer(pipeline, dataset)
sampling = notebook_sampling_explorer(pipeline, dataset)
display(fields, sampling)
```
Виджеты доступны через `.controls`, данные последнего расчёта — `.state`.
Для continuous conditioning можно передать `condition_options={"name": tensor}`.
1D/2D/3D поля поддерживаются для модели точек; у модели траекторий используйте
sampling explorer с отображением стен и концов пути. Свою модель, optimizer и
loss меняйте обычным PyTorch кодом из `03` / `04`.
В Colab при необходимости включите manager стандартных widgets в окружении;
в обычном Jupyter зависимости `[interactive]` включают ipywidgets.


## Радужные траектории денойзинга
Цвет показывает продвижение от исходного шума к финальному сэмплу.
Старт и финиш отмечены отдельно; GIF использует реальную сохранённую историю.


In [ ]:
from pathlib import Path
from IPython.display import Image, display
from diffusion_lab.visualization import save_trajectory_gif

if sampling.state.get("history") is None:
    sampling.generate()
paths = sampling.state["history"][:, :16].permute(1, 0, 2)
gif_path = save_trajectory_gif(
    paths, Path("outputs/notebook_denoising_trajectories.gif"),
    title="Denoising · noise → sample", start_label="Noise x_T", end_label="Sample x_0",
)
display(Image(filename=str(gif_path)))
